***Importações

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import linregress



sns.set_theme(style="whitegrid")
cores = {'Melanoma (C43)': '#e74c3c', 'Não-Melanoma (C44)': '#2c3e50'}

df = pd.read_csv("data/obitos_pele_ceara_2011_2021.csv")

In [ ]:
df_pop = pd.read_csv('data/pop_estimada_IBGE_CE - Página1.csv')
df_pop['População'] = df_pop['População'].str.replace('.', '', regex=False).astype(int)

# Calcular o total de óbitos por Ano e Tipo para fazer as taxas

In [ ]:
df_resumo = df.groupby(['Ano_Obito', 'Tipo_Cancer']).size().reset_index(name='Total_Obitos')
df_taxas = pd.merge(df_resumo, df_pop, left_on='Ano_Obito', right_on='Ano')
df_taxas['Taxa_Mortalidade'] = (df_taxas['Total_Obitos'] / df_taxas['População']) * 100000



# GRÁFICO 1: Taxa de Mortalidade Ajustada (Linhas)

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_taxas, x='Ano_Obito', y='Taxa_Mortalidade', hue='Tipo_Cancer', 
             marker='o', markersize=8, linewidth=2.5, palette=cores)
plt.title("Evolução da Taxa de Mortalidade por Câncer de Pele (2011-2021)", fontsize=14, fontweight='bold')
plt.ylabel("Taxa de Mortalidade (por 100 mil hab.)")
plt.xlabel("Ano do Óbito")
plt.xticks(df_taxas['Ano_Obito'].unique())
plt.legend(title="Tipo")
plt.tight_layout()
plt.savefig('grafico_01_taxa_ajustada.png', dpi=300)
plt.close()

# GRÁFICO 2: Mortalidade Proporcional (100% Stacked Bar)

In [ ]:
df_prop = pd.crosstab(df['Ano_Obito'], df['Tipo_Cancer'], normalize='index') * 100
df_prop.plot(kind='bar', stacked=True, figsize=(10, 6), color=[cores['Melanoma (C43)'], cores['Não-Melanoma (C44)']])
anos = sorted(df['Ano_Obito'].unique())
titulo = f"Proporção de Óbitos por Melanoma e Não-Melanoma ({anos[0]}-{anos[-1]})"
plt.title(titulo, fontsize=14, fontweight='bold')
plt.ylabel("Porcentagem (%)")
plt.xlabel("Ano do Óbito")
plt.legend(title="Tipo de Câncer", loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('grafico_02_proporcao.png', dpi=300)
plt.close()

# GRÁFICO 3: Média de Idade ao Óbito (Boxplot)

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Ano_Obito', y='Idade_Anos', hue='Tipo_Cancer', palette=cores)
plt.title("Distribuição da Idade ao Óbito por Ano", fontsize=14, fontweight='bold')
plt.ylabel("Idade (Anos)")
plt.xlabel("Ano do Óbito")
plt.legend(title="Tipo")
plt.tight_layout()
plt.savefig('grafico_03_idade_boxplot.png', dpi=300)
plt.close()

# =====================================================================
# BLOCO 2: TENDÊNCIA E REGRESSÃO (Aproximação Joinpoint)
# =====================================================================

# GRÁFICO 4: Tendência Não-Melanoma (C44)

In [ ]:
df_c44 = df_taxas[df_taxas['Tipo_Cancer'] == 'Não-Melanoma (C44)']
plt.figure(figsize=(8, 6))
sns.regplot(data=df_c44, x='Ano_Obito', y='Taxa_Mortalidade', color=cores['Não-Melanoma (C44)'], 
            scatter_kws={'s': 50}, line_kws={'color': 'red', 'linewidth': 2})
plt.title("Curva de Tendência de Mortalidade: Não-Melanoma (C44)", fontsize=14, fontweight='bold')
plt.ylabel("Taxa de Mortalidade (por 100 mil hab.)")
plt.xlabel("Ano do Óbito")
plt.xticks(df_taxas['Ano_Obito'].unique())
plt.tight_layout()
plt.savefig('grafico_04_tendencia_C44.png', dpi=300)
plt.close()

# GRÁFICO 5: Tendência Melanoma (C43)

In [ ]:
df_c43 = df_taxas[df_taxas['Tipo_Cancer'] == 'Melanoma (C43)']
plt.figure(figsize=(8, 6))
sns.regplot(data=df_c43, x='Ano_Obito', y='Taxa_Mortalidade', color=cores['Melanoma (C43)'], 
            scatter_kws={'s': 50}, line_kws={'color': 'black', 'linewidth': 2})
plt.title("Curva de Tendência de Mortalidade: Melanoma (C43)", fontsize=14, fontweight='bold')
plt.ylabel("Taxa de Mortalidade (por 100 mil hab.)")
plt.xlabel("Ano do Óbito")
plt.xticks(df_taxas['Ano_Obito'].unique())
plt.tight_layout()
plt.savefig('grafico_05_tendencia_C43.png', dpi=300)
plt.close()

# GRÁFICO 6: Variação Percentual Anual de Óbitos Absolutos

In [ ]:
df_var = df_resumo.pivot(index='Ano_Obito', columns='Tipo_Cancer', values='Total_Obitos').pct_change() * 100
df_var = df_var.dropna().reset_index()
df_var_melt = df_var.melt(id_vars='Ano_Obito', var_name='Tipo_Cancer', value_name='Variacao')
plt.figure(figsize=(10, 6))
sns.barplot(data=df_var_melt, x='Ano_Obito', y='Variacao', hue='Tipo_Cancer', palette=cores)
plt.axhline(0, color='black', linewidth=1)
plt.title("Variação Percentual Anual de Óbitos em Relação ao Ano Anterior", fontsize=14, fontweight='bold')
plt.ylabel("Variação (%)")
plt.xlabel("Ano do Óbito")
plt.legend(title="Tipo")
plt.tight_layout()
plt.savefig('grafico_06_variacao_anual.png', dpi=300)
plt.close()

# GRÁFICO 7: Faixa Etária

In [ ]:
plt.figure(figsize=(10, 6))
ordem_idades = ['0-19 anos', '20-39 anos', '40-59 anos', '60-79 anos', '80+ anos']
sns.countplot(data=df, x='Faixa_Etaria', hue='Tipo_Cancer', palette=cores, order=ordem_idades)
plt.title("Número de Óbitos por Faixa Etária (2011-2021)", fontsize=14, fontweight='bold')
plt.ylabel("Número Total de Óbitos")
plt.xlabel("Faixa Etária")
plt.tight_layout()
plt.savefig('grafico_07_faixa_etaria.png', dpi=300)
plt.close()

# GRÁFICO 8: Sexo (Barras Gerais)

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df[df['Sexo_Nome'] != 'Ignorado'], x='Sexo_Nome', hue='Tipo_Cancer', palette=cores)
plt.title("Óbitos por Câncer de Pele segundo o Sexo (2011-2021)", fontsize=14, fontweight='bold')
plt.ylabel("Número Total de Óbitos")
plt.xlabel("Sexo")
plt.tight_layout()
plt.savefig('grafico_08_sexo.png', dpi=300)
plt.close()

# GRÁFICO 9: Evolução Masculina vs Feminina (C44)

In [ ]:
df_sexo_ano = df[(df['Sexo_Nome'] != 'Ignorado') & (df['Tipo_Cancer'] == 'Não-Melanoma (C44)')]
df_sexo_ano = df_sexo_ano.groupby(['Ano_Obito', 'Sexo_Nome']).size().reset_index(name='Obitos')
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_sexo_ano, x='Ano_Obito', y='Obitos', hue='Sexo_Nome', marker='o', linewidth=2.5, palette=['#3498db', '#e84393'])
plt.title("Evolução dos Óbitos por Não-Melanoma (C44): Homens vs Mulheres", fontsize=14, fontweight='bold')
plt.ylabel("Número de Óbitos")
plt.xlabel("Ano do Óbito")
plt.xticks(df_sexo_ano['Ano_Obito'].unique())
plt.legend(title="Sexo")
plt.tight_layout()
plt.savefig('grafico_09_evolucao_sexo.png', dpi=300)
plt.close()

# GRÁFICO 10: Raça/Cor no C44 (Gráfico de Rosca)

In [ ]:
df_raca = df[(df['Raca_Nome'] != 'Ignorado') & (df['Tipo_Cancer'] == 'Não-Melanoma (C44)')]

contagem_raca = df_raca['Raca_Nome'].value_counts().reset_index()
contagem_raca.columns = ['Raca_Nome', 'Total_Obitos']

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=contagem_raca, 
    x='Total_Obitos', 
    y='Raca_Nome', 
    palette='magma' 
)

plt.title("Letalidade do Não-Melanoma (C44) segundo Raça/Cor (2011-2021)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Número Total de Óbitos", fontsize=12)
plt.ylabel("Raça / Cor", fontsize=12)

for index, row in contagem_raca.iterrows():
    ax.text(row['Total_Obitos'] + 5, index, f"{row['Total_Obitos']} óbitos", 
            color='black', va='center', fontsize=11)

plt.xlim(0, contagem_raca['Total_Obitos'].max() * 1.15)

plt.tight_layout()
plt.savefig('grafico_10_raca_cor_barras.png', dpi=300)
plt.close()

In [ ]:
df_raca_total = df[df['Raca_Nome'] != 'Ignorado']

contagem_raca_tipo = df_raca_total.groupby(['Raca_Nome', 'Tipo_Cancer']).size().reset_index(name='Total_Obitos')

ordem_racas = df_raca_total['Raca_Nome'].value_counts().index

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=contagem_raca_tipo, 
    x='Total_Obitos', 
    y='Raca_Nome', 
    hue='Tipo_Cancer',
    order=ordem_racas,
    palette=cores 
)

plt.title("Óbitos por Câncer de Pele segundo Raça/Cor e Tipo de Neoplasia (2011-2021)", 
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Número Total de Óbitos", fontsize=12)
plt.ylabel("Raça / Cor", fontsize=12)
plt.legend(title="Tipo de Neoplasia", loc='lower right')

for p in ax.patches:
    width = p.get_width()
    if width > 0: 
        ax.text(width + 2, p.get_y() + p.get_height() / 2, f"{int(width)}", 
                color='black', va='center', fontsize=10)

plt.xlim(0, contagem_raca_tipo['Total_Obitos'].max() * 1.15)
plt.tight_layout()
plt.savefig('grafico_10_raca_cor_comparativo.png', dpi=300)
plt.close()

print("Novo Gráfico 10 (Comparativo de Raça C43 vs C44) gerado com sucesso!")

In [ ]:
df_obitos = pd.read_csv('data/obitos_c43_c44_por_ano.csv')
df_pop = pd.read_csv('data/pop_estimada_IBGE_CE - Página1.csv')
df_pop['População'] = df_pop['População'].str.replace('.', '', regex=False).astype(int)


df = pd.merge(df_obitos, df_pop, on='Ano', how='inner')
df = df[(df['Ano'] >= 2011) & (df['Ano'] <= 2021)].copy()


df['Taxa_C43'] = (df['Obitos_C43'] / df['População']) * 100000
df['Taxa_C44'] = (df['Obitos_C44'] / df['População']) * 100000

anos_hist = df['Ano'].values
taxas_c43 = df['Taxa_C43'].values
taxas_c44 = df['Taxa_C44'].values

slope_c43, intercept_c43, _, _, _ = linregress(anos_hist, taxas_c43)
slope_c44, intercept_c44, _, _, _ = linregress(anos_hist, taxas_c44)

anos_proj = np.arange(2022, 2032) # Próximos 10 anos
proj_c43 = slope_c43 * anos_proj + intercept_c43
proj_c44 = slope_c44 * anos_proj + intercept_c44

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

plt.plot(anos_hist, taxas_c43, marker='o', color='#e74c3c', linewidth=2.5, label='Melanoma (C43) - Observado')
plt.plot(anos_hist, taxas_c44, marker='o', color='#2c3e50', linewidth=2.5, label='Não-Melanoma (C44) - Observado')

plt.plot(anos_proj, proj_c43, linestyle='--', color='#e74c3c', linewidth=2.5, label='Melanoma (C43) - Projeção (Tendência)')
plt.plot(anos_proj, proj_c44, linestyle='--', color='#2c3e50', linewidth=2.5, label='Não-Melanoma (C44) - Projeção (Tendência)')

plt.axvspan(2021.5, 2031.5, color='gray', alpha=0.1, label='Período Projetado')

plt.title('Projeção da Mortalidade por Câncer de Pele para os Próximos 10 Anos (2022-2031)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Ano', fontsize=12)
plt.ylabel('Taxa de Mortalidade (por 100 mil hab.)', fontsize=12)
plt.xticks(np.arange(2011, 2032, 2)) 
plt.xlim(2010.5, 2031.5)
plt.legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.savefig('grafico_11_projecao_10_anos.png', dpi=300)
plt.close()

print(f"O gráfico de projeção foi salvo! A inclinação do C44 é {slope_c44:.4f} e a do C43 é {slope_c43:.4f}.")